In [1]:
%load_ext cudf.pandas
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load
import os

if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
### cell 0 ###

data = pd.read_csv(
    "/home/dias-benchmarks/notebooks/josecode1/billionaires-statistics-2023/input/Billionaires Statistics Dataset.csv",
    index_col="rank",
)

In [4]:
### cell 1 ###

factor = 80
data = pd.concat([data] * factor, ignore_index=True)
data.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 211200 entries, 0 to 211199
Data columns (total 34 columns):
 #   Column                                      Non-Null Count   Dtype
---  ------                                      --------------   -----
 0   finalWorth                                  211200 non-null  int64
 1   category                                    211200 non-null  object
 2   personName                                  211200 non-null  object
 3   age                                         206000 non-null  float64
 4   country                                     208160 non-null  object
 5   city                                        205440 non-null  object
 6   source                                      211200 non-null  object
 7   industries                                  211200 non-null  object
 8   countryOfCitizenship                        211200 non-null  object
 9   organization                                26000 non-null   object
 10  selfMade

In [5]:
### cell 2 ###

data.head(5)

,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,organization,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
0,211000,Fashion & Retail,Bernard Arnault & family,74.0,France,Paris,LVMH,Fashion & Retail,France,LVMH Moët Hennessy Louis Vuitton,...,1.1,"$2,715,518,274,227",65.6,102.5,82.5,24.2,60.7,67059887.0,46.227638,2.213749
1,180000,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,Tesla,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
2,114000,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,Amazon,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
3,107000,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,Oracle,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
4,106000,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,Berkshire Hathaway Inc. (Cl A),...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891


In [6]:
### cell 3 ###

data.describe()

,finalWorth,age,birthYear,birthMonth,birthDay,cpi_country,cpi_change_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
count,211200.000000,206000.000000,205120.000000,205120.000000,205120.000000,196480.000000,196480.000000,196640.000000,196720.000000,196640.000000,196560.000000,196640.000000,1.980800e+05,198080.000000,198080.000000
mean,4623.787879,65.140194,1957.183307,5.740250,12.099844,127.755204,4.364169,67.225671,102.858520,78.122823,12.546235,43.963344,5.102053e+08,34.903592,12.583156
std,9832.401495,13.255555,13.279958,3.709371,9.916966,26.447632,3.623035,21.339138,4.710031,3.729350,5.367546,12.142856,5.541342e+08,17.000106,86.745686
min,1000.000000,18.000000,1921.000000,1.000000,1.000000,99.550000,-1.900000,4.000000,84.700000,54.300000,0.100000,9.900000,3.801900e+04,-40.900557,-106.346771
25%,1500.000000,56.000000,1948.000000,2.000000,1.000000,117.240000,1.700000,50.600000,100.200000,77.000000,9.600000,36.600000,6.683440e+07,35.861660,-95.712891
50%,2300.000000,65.000000,1957.000000,6.000000,11.000000,117.240000,2.900000,65.600000,101.800000,78.500000,9.600000,41.200000,3.282395e+08,37.090240,10.451526
75%,4200.000000,75.000000,1966.000000,9.000000,21.000000,125.080000,7.500000,88.200000,102.600000,80.900000,12.800000,59.100000,1.366418e+09,40.463667,104.195397
max,211000.000000,101.000000,2004.000000,12.000000,31.000000,288.570000,53.500000,136.600000,142.100000,84.200000,37.200000,106.300000,1.397715e+09,61.924110,174.885971


In [7]:
### cell 4 ###

country_names = data[
    "country"
].value_counts()  # List of how many billionaires there are in the country

In [8]:
### cell 5 ###

country_names

country
United States               60320
China                       41840
India                       12560
Germany                      8160
United Kingdom               6560
                            ...  
Portugal                       80
Tanzania                       80
Turks and Caicos Islands       80
Uruguay                        80
Uzbekistan                     80
Name: count, Length: 78, dtype: int64

> > > > >  ****10 countries with the most billionaires****

In [9]:
### cell 6 ###

data_100 = data.loc[:100, ["finalWorth", "category", "country"]]

In [10]:
### cell 7 ###

data_100_category = data_100["category"].value_counts()

> > > > > ****Most Richest 100 person in the world and their categories****

In [11]:
### cell 8 ###

data_usa = data[
    data["country"] == "United States"
]  # We focus on billionaires based in United States

In [12]:
### cell 9 ###

data_usa_category = data_usa["category"].value_counts()
data_usa_category

category
Finance & Investments         15200
Technology                    11280
Food & Beverage                5920
Fashion & Retail               5040
Real Estate                    4080
Media & Entertainment          2960
Energy                         2800
Sports                         2800
Healthcare                     2720
Manufacturing                  2240
Service                        1520
Diversified                    1040
Automotive                      960
Gambling & Casinos              480
Construction & Engineering      400
Logistics                       400
Telecom                         400
Metals & Mining                  80
Name: count, dtype: int64

> > > > > ****All Billionaires in the USA and their categories****

In [13]:
%%time
### cell 10 ###

data_usa_city = data_usa["city"].value_counts()
print(data_usa_city)
data_usa_city.info()

city
New York            7920
San Francisco       2960
Los Angeles         2720
Palm Beach          1680
Dallas              1600
                    ... 
Windermere            80
Wynnewood             80
Yorktown Heights      80
Youngstown            80
Ypsilanti             80
Name: count, Length: 268, dtype: int64
<class 'pandas.core.series.Series'>
Index: 268 entries, New York to Ypsilanti
Series name: count
Non-Null Count  Dtype
--------------  -----
268 non-null    int64
dtypes: int64(1)
memory usage: 4.2+ KB
CPU times: user 17.5 ms, sys: 4.93 ms, total: 22.4 ms
Wall time: 21.6 ms


268 city exist, I'll try .head(20)

> > > > > **The 20 cities has most billionaires in usa**

In [14]:
### cell 11 ###

data.head(5)

,finalWorth,category,personName,age,country,city,source,industries,countryOfCitizenship,organization,...,cpi_change_country,gdp_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
0,211000,Fashion & Retail,Bernard Arnault & family,74.0,France,Paris,LVMH,Fashion & Retail,France,LVMH Moët Hennessy Louis Vuitton,...,1.1,"$2,715,518,274,227",65.6,102.5,82.5,24.2,60.7,67059887.0,46.227638,2.213749
1,180000,Automotive,Elon Musk,51.0,United States,Austin,"Tesla, SpaceX",Automotive,United States,Tesla,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
2,114000,Technology,Jeff Bezos,59.0,United States,Medina,Amazon,Technology,United States,Amazon,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
3,107000,Technology,Larry Ellison,78.0,United States,Lanai,Oracle,Technology,United States,Oracle,...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891
4,106000,Finance & Investments,Warren Buffett,92.0,United States,Omaha,Berkshire Hathaway,Finance & Investments,United States,Berkshire Hathaway Inc. (Cl A),...,7.5,"$21,427,700,000,000",88.2,101.8,78.5,9.6,36.6,328239523.0,37.090240,-95.712891


In [17]:
%%cudf.pandas.profile
data.finalWorth = data.finalWorth.astype(float)
data_grouped = data.groupby('category').finalWorth.mean()

                                                                                                         
                                        Total time elapsed: 0.110 seconds                                
                                      6 GPU function calls in 0.026 seconds                              
                                      0 CPU function calls in 0.000 seconds                              
                                                                                                         
                                                      Stats                                              
                                                                                                         
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function            ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.__getattr__ │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype      │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.__setattr__ │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.groupby   │ 1          │ 0.001       │ 0.001       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.__getattr__ │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ GroupBy.mean        │ 1          │ 0.015       │ 0.015       │ 0          │ 0.000       │ 0.000       │
└─────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [20]:
%%cudf.pandas.profile
data.finalWorth = data.finalWorth.astype(float)
worth_average = []
for i in category_list:
    x = data[data['category']==i]
    worth_ratio = sum(x.finalWorth)/len(x)
    worth_average.append(worth_ratio)

                                                                                                           
                                        Total time elapsed: 13.834 seconds                                 
                                      93 GPU function calls in 12.434 seconds                              
                                       0 CPU function calls in 0.000 seconds                               
                                                                                                           
                                                       Stats                                               
                                                                                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function              ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ NDFrame.__getattr__   │ 19         │ 11.870      │ 0.625       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.astype        │ 1          │ 0.002       │ 0.002       │ 0          │ 0.000       │ 0.000       │
│ NDFrame.__setattr__   │ 1          │ 0.003       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__getitem__ │ 36         │ 0.494       │ 0.014       │ 0          │ 0.000       │ 0.000       │
│ OpsMixin.__eq__       │ 18         │ 0.055       │ 0.003       │ 0          │ 0.000       │ 0.000       │
│ DataFrame.__len__     │ 18         │ 0.009       │ 0.001       │ 0          │ 0.000       │ 0.000       │
└───────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

In [19]:
%%cudf.pandas.profile
data.corr(numeric_only=True)

,finalWorth,age,selfMade,birthYear,birthMonth,birthDay,cpi_country,cpi_change_country,gross_tertiary_education_enrollment,gross_primary_education_enrollment_country,life_expectancy_country,tax_revenue_country_country,total_tax_rate_country,population_country,latitude_country,longitude_country
finalWorth,1.000000,0.067053,-0.023831,-0.066721,0.003407,0.059315,-0.042842,0.035702,0.066711,-0.008880,0.021819,-0.009270,-0.036381,-0.053024,0.031122,-0.101048
age,0.067053,1.000000,-0.050538,-0.999336,0.015322,0.081547,-0.001479,0.115669,0.061736,0.066394,0.020327,0.006429,-0.151771,-0.167812,-0.122544,-0.169338
selfMade,-0.023831,-0.050538,1.000000,0.050333,0.001391,-0.030345,-0.015086,0.031552,0.012241,-0.224357,-0.052792,-0.156087,0.111429,0.221644,0.070572,0.106552
birthYear,-0.066721,-0.999336,0.050333,1.000000,-0.045066,-0.091512,-0.000738,-0.118060,-0.061316,-0.066810,-0.017936,-0.005262,0.151297,0.167355,0.125035,0.169756
birthMonth,0.003407,0.015322,0.001391,-0.045066,1.000000,0.221384,0.056870,0.106427,0.049269,0.026174,-0.044165,0.001484,-0.046784,-0.050506,-0.038060,-0.062697
birthDay,0.059315,0.081547,-0.030345,-0.091512,0.221384,1.000000,0.037517,0.146357,0.171608,0.045075,0.004498,0.034128,-0.149580,-0.204271,0.006617,-0.188214
cpi_country,-0.042842,-0.001479,-0.015086,-0.000738,0.056870,0.037517,1.000000,0.436769,-0.456428,0.279601,-0.747716,-0.037022,0.245961,0.218303,-0.215101,0.258661
cpi_change_country,0.035702,0.115669,0.031552,-0.118060,0.106427,0.146357,0.436769,1.000000,0.167455,0.053483,-0.393884,-0.317516,0.003272,0.066501,-0.113692,-0.470460
gross_tertiary_education_enrollment,0.066711,0.061736,0.012241,-0.061316,0.049269,0.171608,-0.456428,0.167455,1.000000,-0.298473,0.523931,0.028123,-0.393902,-0.543031,0.122088,-0.578740
gross_primary_education_enrollment_country,-0.008880,0.066394,-0.224357,-0.066810,0.026174,0.045075,0.279601,0.053483,-0.298473,1.000000,-0.306788,0.129724,0.098344,0.009805,-0.199590,-0.009239


                                                                                                        
                                       Total time elapsed: 0.798 seconds                                
                                     1 GPU function calls in 0.100 seconds                              
                                     1 CPU function calls in 0.515 seconds                              
                                                                                                        
                                                     Stats                                              
                                                                                                        
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Function           ┃ GPU ncalls ┃ GPU cumtime ┃ GPU percall ┃ CPU ncalls ┃ CPU cumtime ┃ CPU percall ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ DataFrame.corr     │ 0          │ 0.000       │ 0.000       │ 1          │ 0.515       │ 0.515       │
│ DataFrame.__repr__ │ 1          │ 0.100       │ 0.100       │ 0          │ 0.000       │ 0.000       │
└────────────────────┴────────────┴─────────────┴─────────────┴────────────┴─────────────┴─────────────┘

Not all pandas operations ran on the GPU. The following functions required CPU fallback:

- DataFrame.corr

To request GPU support for any of these functions, please file a Github issue here: 
]8;id=908130;https://github.com/rapidsai/cudf/issues/new?assignees=&labels=%3F+-+Needs+Triage%2C+feature+request&projects=&template=pandas_function_request.md&title=%5BFEA%5D\https://github.com/rapidsai/cudf/issues/new/choose]8;;\.

In [15]:
### cell 12 ###

category_list = list(data.category.unique())
category_list

['Fashion & Retail',
 'Automotive',
 'Technology',
 'Finance & Investments',
 'Media & Entertainment',
 'Telecom',
 'Diversified',
 'Food & Beverage',
 'Logistics',
 'Gambling & Casinos',
 'Manufacturing',
 'Real Estate',
 'Metals & Mining',
 'Energy',
 'Healthcare',
 'Service',
 'Construction & Engineering',
 'Sports']

In [ ]:
### cell 13 ###

data["finalWorth"] = data["finalWorth"].astype("float")

# Compute the average finalWorth per category using groupby
data2 = data.groupby("category", as_index=False).agg({"finalWorth": "mean"})

# Rename the aggregated column to worth_average for clarity
data2 = data2.rename(columns={"finalWorth": "worth_average"})

# Sort the results by worth_average in descending order
new_data = data2.sort_values(by="worth_average", ascending=False)

In [ ]:
### cell 14 ###

data3 = data.dropna(
    axis="index", how="any", inplace=False
)  # ı created new df because so many nan values was exist

In [ ]:
### cell 15 ###

data3.head()

In [ ]:
### cell 16 ###

data3.info()  # in the new df we have 238 rows for each column

In [ ]:
### cell 17 ###

data3.countryOfCitizenship.unique()  # and 238 billionaires in usa, others deleted.